# Entrenar YOLO26 para UNEMI Guide AI (Roboflow + Colab)

Este cuaderno entrena el modelo que reconoce **en qué zona del campus** está el estudiante a partir de una foto.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

Ejecuta las celdas **en orden** (botón ▶ de cada una). Al final se descarga `modelo_unemi.zip` con `best.pt` y `data.yaml`: eso es lo que se instala en la app.

Guía completa: `docs/entrenar_yolo_roboflow.md` del proyecto.

## 1. Comprobar la GPU e instalar las librerías

In [ ]:
!nvidia-smi -L || echo "⚠️ SIN GPU: ve a Entorno de ejecución → Cambiar tipo → GPU (T4)"
!pip install -q -U ultralytics roboflow

## 2. Descargar tus fotos etiquetadas desde Roboflow

En Roboflow: tu proyecto → **Versions** → la versión que generaste → **Download Dataset** → formato **YOLO26** → **Show download code** → copia el código de *Jupyter*.

Pega ese código **en la celda de abajo**, reemplazando las líneas de ejemplo. Debe terminar creando la variable `dataset`.

In [ ]:
# ⬇️ BORRA estas líneas de ejemplo y PEGA aquí el código que te da Roboflow ⬇️
from roboflow import Roboflow
rf = Roboflow(api_key="PEGA_AQUI_TU_API_KEY")
project = rf.workspace("unemi-guide-ai").project("unemi-zonas")
version = project.version(1)
dataset = version.download("yolov26")  # usa el formato que diga Roboflow

print("Fotos descargadas en:", dataset.location)

## 3. Revisar que los nombres de las clases sean zonas de la app

Roboflow quita tildes y paréntesis (`Bloque T -FACI-`); la app los reconoce igual. Si sale ❌, la clase no corresponde a ninguna zona: corrígela en Roboflow (*Classes & Tags*), genera una versión nueva y repite el paso 2.

In [ ]:
import re, unicodedata, yaml

# Clases del proyecto de Roboflow "unemi-zonas" (sin tildes ni paréntesis).
CLASES_DE_LA_APP = [
    "Bloque B",
    "Bloque C",
    "Bloque E",
    "Bloque F",
    "Bloque H",
    "Bloque I",
    "Bloque J",
    "Bloque K",
    "Bloque L",
    "Bloque M",
    "Bloque N",
    "Bloque O",
    "Bloque P",
    "Bloque R",
    "Bloque S",
    "Bloque T -FACI-",
    "Bloque U",
    "Bloques V-W-X",
    "CRAI",
    "Posgrado",
    "SIM",
    "CDIP",
    "Bloque de Investigacion de Biotecnologia",
    "Estadio Universitario UNEMI",
    "Piscina UNEMI",
    "Entrada de vehiculos",
    "Area verde -Bloques R-U-",
    "Estacionamiento 1",
    "Estacionamiento 2"
]

def clave(nombre):
    """Igual que la app: ignora tildes, signos, espacios y mayúsculas."""
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", sin_tildes.lower())

CLAVES = {clave(n) for n in CLASES_DE_LA_APP}

with open(f"{dataset.location}/data.yaml", encoding="utf-8") as f:
    datos = yaml.safe_load(f)
nombres = datos["names"].values() if isinstance(datos["names"], dict) else datos["names"]

todo_bien = True
for nombre in nombres:
    if clave(nombre) in CLAVES:
        print("✅", nombre)
    else:
        todo_bien = False
        print("❌", repr(nombre), "-> no es una zona de la app. Revisa cómo está escrito.")
print()
print("Todo listo para entrenar 👍" if todo_bien else "⚠️ Corrige los nombres con ❌ antes de entrenar.")

## 3b. ¿Cuántas fotos tiene cada zona?

Cuenta las fotos de *train* por zona y las fotos **Nada** (marcadas como *Null* en Roboflow: no son ninguna zona, sin recuadro).

- ⚠️ Zona con **menos de 40** fotos → el modelo la reconocerá mal: toma más fotos o déjala para el próximo entrenamiento.
- Las fotos **Nada** deberían ser entre el **5 % y el 10 %** del total: así el modelo aprende a decir "no sé" en vez de adivinar.

In [ ]:
import os, glob
from collections import Counter

nombres_lista = list(nombres)
fotos_por_zona = Counter()
fotos_nada = 0
etiquetas = glob.glob(f"{dataset.location}/train/labels/*.txt")
total_train = len(glob.glob(f"{dataset.location}/train/images/*"))
for archivo in etiquetas:
    with open(archivo) as f:
        clases = {int(l.split()[0]) for l in f if l.strip()}
    if not clases:
        fotos_nada += 1
    for c in clases:
        fotos_por_zona[nombres_lista[c]] += 1
fotos_nada += total_train - len(etiquetas)  # fotos Null sin archivo de etiquetas

for zona in sorted(nombres_lista, key=lambda z: fotos_por_zona[z]):
    n = fotos_por_zona[zona]
    marca = "⚠️" if n < 40 else "✅"
    print(f"{marca} {zona:42s} {n:4d} fotos")
print()
porcentaje = fotos_nada / max(total_train, 1)
print(f"Fotos Nada (Null): {fotos_nada} de {total_train} ({porcentaje:.0%})",
      "✅" if 0.05 <= porcentaje <= 0.15 else "⚠️ ideal: 5-10 %")

## 4. Entrenar (30–90 minutos)

`yolo26n.pt` es el modelo más pequeño: el servidor tiene poca memoria. Si el entrenamiento deja de mejorar, se detiene solo (`patience`).

- `fliplr=0.0`: **no** se voltean las fotos como espejo; los letreros ("Bloque N", "CRAI") quedarían al revés y el modelo confundiría bloques.
- `close_mosaic=15`: las últimas épocas usan fotos completas, como las que toma el estudiante.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
resultados = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=150,
    imgsz=640,
    batch=-1,          # elige el tamaño de lote según la memoria de la GPU
    patience=30,
    fliplr=0.0,        # sin espejo: los letreros deben leerse bien
    degrees=5,         # celular un poco inclinado
    hsv_v=0.4,         # mañana / tarde / nublado
    close_mosaic=15,
    name="unemi_zonas",
    exist_ok=True,
)

## 5. ¿Quedó bien?

- **mAP50 > 0.8** → bueno. Entre 0.6 y 0.8 → aceptable para una demo. Menos de 0.6 → faltan fotos o están mal etiquetadas.
- En la **matriz de confusión** mira qué zonas confunde entre sí: a esas les faltan fotos.

In [ ]:
from IPython.display import Image, display
carpeta = "runs/detect/unemi_zonas"
metricas = YOLO(f"{carpeta}/weights/best.pt").val(data=f"{dataset.location}/data.yaml", verbose=False)
print(f"mAP50 = {metricas.box.map50:.2f}   |   mAP50-95 = {metricas.box.map:.2f}")
display(Image(f"{carpeta}/results.png", width=900))
display(Image(f"{carpeta}/confusion_matrix.png", width=700))

## 6. Probar con las fotos de *test* (fotos que el modelo nunca vio)

Así funciona la app: toma la detección con **más confianza** y, si es menor a **0.65**, dice "no reconocida" (semáforo rojo).
Esta celda mide el **acierto de zona** de esa misma forma y lista las fotos en las que falló.

- Acierto **> 85 %** → listo para la app. Entre 70 % y 85 % → demo. Menos → faltan fotos de las zonas que aparecen en los errores.

In [ ]:
import glob, os
from collections import Counter

UMBRAL_APP = 0.65   # igual que vision/utils.py (UMBRAL_AMARILLO)
modelo = YOLO(f"{carpeta}/weights/best.pt")
fotos = sorted(glob.glob(f"{dataset.location}/test/images/*"))

def zona_real(foto):
    etiqueta = foto.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if not os.path.exists(etiqueta):
        return None
    with open(etiqueta) as f:
        clases = [int(l.split()[0]) for l in f if l.strip()]
    return modelo.names[Counter(clases).most_common(1)[0][0]] if clases else None

aciertos, errores = 0, []
for foto in fotos:
    r = modelo(foto, conf=0.25, verbose=False)[0]
    predicha, confianza = None, 0.0
    if len(r.boxes):
        mejor = int(r.boxes.conf.argmax())
        confianza = float(r.boxes.conf[mejor])
        if confianza >= UMBRAL_APP:
            predicha = r.names[int(r.boxes.cls[mejor])]
    real = zona_real(foto)
    if predicha == real:
        aciertos += 1
    else:
        errores.append((os.path.basename(foto), real or "Nada", predicha or "no reconocida", confianza))

print(f"Acierto de zona: {aciertos}/{len(fotos)} = {aciertos / max(len(fotos), 1):.0%}\n")
for nombre, real, predicha, confianza in errores:
    print(f"❌ {nombre[:35]:35s} era {real:30s} -> {predicha} ({confianza:.0%})")

# Muestra 4 fotos con sus recuadros
for i, foto in enumerate(fotos[:4]):
    r = modelo(foto, conf=0.25, verbose=False)[0]
    display(Image(r.save(filename=f"/content/prueba_{i}.jpg"), width=500))

## 7. Descargar el modelo

Se descarga **`modelo_unemi.zip`**. Envíalo junto con 3–5 fotos del campus que **no** hayas subido a Roboflow, para probarlo antes de instalarlo en la app.

In [ ]:
import shutil, os
os.makedirs("modelo_unemi", exist_ok=True)
shutil.copy(f"{carpeta}/weights/best.pt", "modelo_unemi/best.pt")
shutil.copy(f"{dataset.location}/data.yaml", "modelo_unemi/data.yaml")
for extra in ["results.png", "confusion_matrix.png"]:
    if os.path.exists(f"{carpeta}/{extra}"):
        shutil.copy(f"{carpeta}/{extra}", f"modelo_unemi/{extra}")
shutil.make_archive("modelo_unemi", "zip", "modelo_unemi")

from google.colab import files
files.download("modelo_unemi.zip")